# Particle accelerators: protons and alpha particles

Nuclear physics needs fast charged particles: to get a proton or an alpha particle (a helium-4 nucleus) close enough
to a nucleus to react, it has to climb the Coulomb barrier, a few MeV for light nuclei and over 20 MeV for gold. This
notebook builds the classic machines that do it and checks each against its textbook formula:

1. an **electrostatic accelerator** (Cockcroft–Walton, Van de Graaff): one big voltage, energy $qV$;
2. a **drift-tube linac** (Wideröe, Alvarez): many RF gaps, and *phase stability*, which keeps a bunch together;
3. a **cyclotron** (Lawrence): a magnetic field bends the beam back through the same gap every turn;
4. **relativity**, which detunes the classical cyclotron, and the isochronous cyclotron that fixes it;
5. **Rutherford scattering**, the experiment that showed why accelerators were needed.

The engine has no built-in units, so sections 1–4 use SI (metres, seconds, kilograms, coulombs, volts, tesla) and
convert energies to MeV for display; section 5 uses nuclear units (fm, MeV). Fields are external: `ElectricField`
and `MagneticField` for uniform ones, `FieldForce` for fields given as Python functions of time and position. The
engine's dynamics are Newtonian (special relativity is planned, see `backlog/30-special-relativity.md`); section 4
adds it with a `CustomForce`.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import physim as ps

e = 1.602176634e-19        # elementary charge, C
c = 299792458.0            # speed of light, m/s
MeV = 1e6 * e              # J

# Ions: (charge in e, rest energy in MeV), CODATA 2018
IONS = {"proton": (1, 938.27208816), "deuteron": (1, 1875.61294257), "alpha": (2, 3727.3794066)}

def charge(ion): return IONS[ion][0] * e
def mass(ion): return IONS[ion][1] * MeV / c**2

def kinetic_MeV(vel, m):
    # Newtonian kinetic energy of each particle in MeV; vel is (..., 3)
    return 0.5 * m * np.sum(vel**2, axis=-1) / MeV

## 1. Electrostatic acceleration

In 1932 Cockcroft and Walton split lithium with protons from a voltage multiplier of a few hundred kV, the first
nuclear reaction driven by an accelerator. The physics of an electrostatic column is just a uniform field
$E = V/d$ along the tube: an ion released at rest gains exactly $qV$ whatever its mass, while its transit time
$d\sqrt{2m/qV}$ grows with the mass. An alpha particle (charge $2e$) gains twice the proton's energy.

Each ion's arrival at the end of the column is found with an event, located to round-off inside a step, so its energy
is read off exactly there rather than at the nearest recorded frame. The ions do not interact (there is no
`Coulomb` force between them), so one world carries all three.

In [ ]:
V, d = 700e3, 2.0                      # 700 kV over a 2 m column
ions = ["proton", "deuteron", "alpha"]
w = ps.World("verlet")
for ion in ions:
    w.add_particle([0, 0, 0], mass=mass(ion), charge=charge(ion))
w.add_force(ps.ElectricField([V / d, 0, 0]))

arrive = [ps.Event.coordinate(k, "x", value=d, direction=+1) for k in range(len(ions))]
tr = w.run(1e-10, 6000, record_every=20, events=arrive)

for k, ion in enumerate(ions):
    hit = tr.event_index == k
    T = kinetic_MeV(tr.event_vel[hit][0, k], mass(ion))
    print(f"{ion:>8}: T = {T:.6f} MeV (qV = {IONS[ion][0] * V / 1e6:.6f}), "
          f"transit {tr.event_t[hit][0] * 1e9:6.1f} ns (theory {d * math.sqrt(2 * mass(ion) / (charge(ion) * V)) * 1e9:6.1f})")
print(f"total energy (kinetic + potential in the field) conserved to {np.ptp(tr.energy) / tr.kinetic[-1]:.0e}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
styles = [dict(lw=3, alpha=0.5), dict(lw=1.5), dict(lw=1.5, ls="--")]
for k, ion in enumerate(ions):
    inside = tr.pos[:, k, 0] <= d
    T = kinetic_MeV(tr.vel[inside, k], mass(ion))
    a1.plot(tr.pos[inside, k, 0], T, color=ps.plot.color(k), label=ion, **styles[k])
    a2.plot(tr.t[inside] * 1e9, T, color=ps.plot.color(k), label=ion, **styles[k])
a1.set_xlabel("position along the column (m)"); a1.set_ylabel("kinetic energy (MeV)")
a1.set_title(r"energy gained $= qEx$: set by the charge", fontsize=10)
a2.set_xlabel("time (ns)"); a2.set_ylabel("kinetic energy (MeV)")
a2.set_title(r"energy vs time $= (qEt)^2/2m$: set by $q^2/m$", fontsize=10)
ps.plot.tidy(a1, grid=True); ps.plot.tidy(a2, grid=True);

On the left the proton and deuteron coincide (same charge); on the right the proton and alpha nearly coincide,
because $q^2/m$ is almost the same for both. A single voltage stops being practical at a few MV, where the column sparks over. Every higher-energy machine instead
reuses a modest voltage many times, timed with a radio-frequency (RF) source.

## 2. Drift-tube linac and phase stability

A Wideröe linac is a row of metal tubes connected alternately to the two sides of an RF generator. Inside a tube there
is no field; in each gap between tubes the field is $\pm (V_0/\text{gap}) \cos\omega t$, with the sign alternating
from gap to gap. A proton that crosses a gap while the field pushes it forward drifts through the next tube while
the field reverses, and arrives at the next gap to be pushed forward again. That requires a tube length of half an RF
period at the proton's current speed, $L_n = v_n / 2f$, so the tubes get longer down the machine.

The *synchronous* proton crosses every gap at the same RF phase $\phi_s$ and gains $qV_0 T\cos\phi_s$ per gap, where
$T = e^{-(\omega\sigma/v)^2/2}$ is the transit-time factor of a Gaussian gap of width $\sigma$. The design below
places the gaps for that proton. The choice of $\phi_s$ matters. With $\phi_s < 0$ (the field still rising as the
synchronous proton crosses), a proton that arrives late sees a stronger field, gains more, speeds up and catches up;
an early one gains less and falls back. Protons near the synchronous phase oscillate about it and stay in the
bunch. With $\phi_s > 0$ the same argument runs backwards and the bunch flies apart. Here a bunch is launched
spread over $\pm 90°$ of RF phase around $\phi_s = -30°$.

In [ ]:
f_rf, V0, sigma = 50e6, 150e3, 5e-3     # 50 MHz, 150 kV per gap, gap field Gaussian of width 5 mm
n_gaps, W0 = 40, 0.5                    # 40 gaps, 0.5 MeV protons from an injector
omega = 2 * np.pi * f_rf
qp, mp = charge("proton"), mass("proton")

def design(phi_s):
    # gap centres for a proton crossing every gap at phase phi_s; also its energy after each gap
    z, W, gaps, energies = 0.0, W0, [], []
    for n in range(n_gaps):
        gaps.append(z)
        v = math.sqrt(2 * W * MeV / mp)
        W += qp * V0 * math.exp(-0.5 * (omega * sigma / v) ** 2) * math.cos(phi_s) / MeV
        energies.append(W)
        z += math.sqrt(2 * W * MeV / mp) / (2 * f_rf)    # drift tube: half an RF period at the new speed
    return np.array(gaps), np.array(energies)

phi_s = math.radians(-30)
z_gap, W_design = design(phi_s)
print(f"linac length {z_gap[-1]:.2f} m; first tube {np.diff(z_gap)[0] * 100:.1f} cm, last {np.diff(z_gap)[-1] * 100:.1f} cm; "
      f"design energy {W0} -> {W_design[-1]:.3f} MeV")

In [ ]:
polarity = (-1.0) ** np.arange(n_gaps)
peak = V0 / (math.sqrt(2 * math.pi) * sigma)        # so each gap's field integrates to V0
v_inj = math.sqrt(2 * W0 * MeV / mp)
z_start = -5 * sigma                                # just upstream of the first gap's field
psi = phi_s - omega * (-z_start / v_inj)            # RF phase offset: the reference proton meets gap 0 at phi_s

def rf_gaps(t, pos):
    z = pos[:, 2:3]
    E = np.zeros_like(pos)
    E[:, 2] = math.cos(omega * t + psi) * np.sum(polarity * peak * np.exp(-0.5 * ((z - z_gap) / sigma) ** 2), axis=1)
    return E

launch = phi_s + np.radians(np.linspace(-90, 90, 37))   # RF phase at which each proton would reach gap 0
w = ps.World("yoshida4")
for p in launch:                                    # a later phase = further back along the beam line
    w.add_particle([0, 0, z_start - v_inj * (p - phi_s) / omega], [0, 0, v_inj], mass=mp, charge=qp)
w.add_force(ps.FieldForce(E=rf_gaps, name="RF gaps"))

t_end = (n_gaps + 4) / (2 * f_rf)
tr = w.run(2e-11, round(t_end / 2e-11), record_every=5, energies=False)

# RF phase at which each proton crosses each gap centre, relative to the gap's accelerating crest
phase = np.full((len(launch), n_gaps), np.nan)
for k in range(len(launch)):
    z = tr.pos[:, k, 2]
    if np.all(np.diff(z) > 0):
        t_cross = np.interp(z_gap, z, tr.t, right=np.nan)
        wrapped = np.angle(np.exp(1j * (omega * t_cross + psi - np.pi * np.arange(n_gaps))))
        ok = np.isfinite(wrapped)
        phase[k, ok] = np.unwrap(wrapped[ok])     # lost protons slip by whole RF periods
W_final = kinetic_MeV(tr.vel[-1], mp)
captured = np.abs(W_final - W_design[-1]) < 0.3
ref = np.argmin(np.abs(launch - phi_s))
print(f"reference proton: {W_final[ref]:.3f} MeV (design {W_design[-1]:.3f}), "
      f"phase stays within {np.degrees(np.nanmax(np.abs(phase[ref] - phi_s))):.1f} deg of phi_s")
print(f"captured {captured.sum()} of {len(launch)} protons, launched at phases between "
      f"{np.degrees(launch[captured].min()):.0f} and {np.degrees(launch[captured].max()):.0f} deg")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for k in range(len(launch)):
    col = ps.plot.color(0) if captured[k] else "0.7"
    a1.plot(np.arange(n_gaps), np.degrees(phase[k]), color=col, lw=0.8, zorder=2 if captured[k] else 1)
a1.axhline(math.degrees(phi_s), color=ps.plot.color(1), lw=1.5, ls="--", label=r"synchronous phase $\phi_s$")
a1.set_xlabel("gap"); a1.set_ylabel("RF phase at the gap (deg)"); a1.set_ylim(-180, 180)
a1.set_title("phase oscillations: captured (blue) and lost (grey)", fontsize=10)
ps.plot.tidy(a1)

a2.scatter(np.degrees(launch), W_final, s=14, c=[ps.plot.color(0) if k else "0.6" for k in captured])
a2.axhline(W_design[-1], color=ps.plot.color(1), lw=1, ls="--", label="design energy")
a2.set_xlabel("RF phase at the first gap (deg)"); a2.set_ylabel("final energy (MeV)")
ps.plot.tidy(a2, grid=True);

The captured protons oscillate about $\phi_s$ (these are *synchrotron oscillations*; every RF accelerator has them)
and all leave at close to the design energy. The capture range, from about $-70°$ to $+30°$ here, is the RF *bucket*:
the separatrix runs from $+|\phi_s|$ to about $-2|\phi_s|$. Protons outside it slip out of step and leave with
whatever they happened to gain. Note the asymmetry: a proton launched at $+40°$ is lost even though it starts on the
accelerating side of the crest, because it sits on the unstable, falling slope.

## 3. The cyclotron

Lawrence's idea (1931) folds the linac into a spiral. A uniform field $B$ bends an ion into a circle at the
*cyclotron frequency* $\omega_c = qB/m$, which (non-relativistically) does not depend on its speed. Two hollow
D-shaped electrodes ("dees") with a gap between them carry an RF voltage at exactly $\omega_c$, so the ion is
accelerated each time it crosses the gap, twice a turn, gaining $2qV_0$ per turn, and spirals outwards. It leaves at
the radius $R$ of the magnet with
$$T = \frac{(qBR)^2}{2m}.$$
An alpha particle has half the proton's $q/m$, so it needs half the RF frequency, and its $q^2/m$ is about the same, so
it leaves with about the same energy.

The `boris` integrator rotates each velocity about $\mathbf B$ exactly, so the gyration is exact apart from a tiny
phase error, $(\omega h)^2/12$ per step. The gap field is a `FieldForce`. A terminal event stops the run at
$r = R$, where a real machine would extract the beam.

In [ ]:
B, R, V_dee, gap = 1.5, 0.5, 100e3, 0.02     # 1.5 T, 0.5 m pole radius, 100 kV dees, 2 cm gap field

def cyclotron(ion, steps_per_turn=500):
    q, m = charge(ion), mass(ion)
    w_c = q * B / m
    peak = V_dee / (math.sqrt(2 * math.pi) * gap)

    def dee_gap(t, pos):                       # the gap lies along the y axis
        E = np.zeros_like(pos)
        E[:, 0] = peak * np.exp(-0.5 * (pos[:, 0] / gap) ** 2) * np.cos(w_c * t)
        return E

    w = ps.World("boris")
    w.add_particle([0, 0, 0], mass=m, charge=q)
    w.add_force(ps.MagneticField([0, 0, B]))
    w.add_force(ps.FieldForce(E=dee_gap, name="dee gap"))
    extract = ps.Event(lambda t, pos, vel, mass: math.hypot(pos[0, 0], pos[0, 1]) - R, direction=+1, terminal=True)
    period = 2 * np.pi / w_c
    tr = w.run(period / steps_per_turn, steps_per_turn * 400, record_every=10, events=[extract], energies=False)
    return tr, period

runs = {}
for ion in ["proton", "alpha"]:
    tr, period = cyclotron(ion)
    runs[ion] = tr, period
    T = kinetic_MeV(tr.vel[-1, 0], mass(ion))
    print(f"{ion:>6}: RF {1 / period / 1e6:6.3f} MHz, extracted after {tr.t[-1] / period:5.1f} turns "
          f"with {T:.2f} MeV; (qBR)^2/2m = {(charge(ion) * B * R) ** 2 / (2 * mass(ion)) / MeV:.2f} MeV, "
          f"energy gain per turn {2 * IONS[ion][0] * V_dee / 1e6:.1f} MeV")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.6))
tr, period = runs["proton"]
first = tr.t <= 12 * period
a1.plot(tr.pos[first, 0, 0], tr.pos[first, 0, 1], lw=0.8, color=ps.plot.color(0))
a1.axvspan(-gap, gap, color=ps.plot.color(1), alpha=0.15, label="accelerating gap")
a1.text(-0.09, 0.0, "dee", ha="center", color="0.4"); a1.text(0.09, 0.0, "dee", ha="center", color="0.4")
a1.set_aspect("equal"); a1.set_xlabel("x (m)"); a1.set_ylabel("y (m)")
a1.set_title("proton: the first 12 turns", fontsize=10)
a1.legend(frameon=False, loc="upper right"); ps.plot.tidy(a1)

r = np.linspace(0, R, 200)
for k, ion in enumerate(["proton", "alpha"]):
    tr, period = runs[ion]
    radius = np.hypot(tr.pos[:, 0, 0], tr.pos[:, 0, 1])
    a2.plot(radius, kinetic_MeV(tr.vel[:, 0], mass(ion)), lw=[2.5, 1][k], alpha=[0.5, 1][k], color=ps.plot.color(k), label=ion)
    a2.plot(r, (charge(ion) * B * r) ** 2 / (2 * mass(ion)) / MeV, color=ps.plot.color(k), lw=1.5, ls="--", alpha=0.7)
a2.set_xlabel("distance from centre (m)"); a2.set_ylabel("kinetic energy (MeV)")
a2.set_title(r"energy vs radius; dashed: $(qBr)^2/2m$", fontsize=10)
ps.plot.tidy(a2, grid=True);

The orbit is a spiral of half circles whose radius jumps at each gap crossing, by a lot at first and less as the ion
speeds up ($r \propto \sqrt T$). The energy follows $(qBr)^2/2m$ at every radius; the proton and alpha curves overlap
because $q^2/m$ is nearly the same for both. The extracted energy differs from the formula at $r = R$ only by where in its last half turn the ion happens to cross $R$. In this
Newtonian world, the ion stays exactly in step with the RF for ever.

## 4. Relativity detunes the cyclotron

At 27 MeV a proton moves at 0.23 $c$, and its true revolution frequency is $qB/\gamma m$, about 3% below the RF. The
lag in RF phase builds up turn after turn until the ion crosses the gap when the field is reversed and starts losing
energy. That limited classical cyclotrons to about 20 MeV of protons.

The engine integrates Newtonian dynamics, but the relativistic equation of motion can be written as an ordinary
acceleration of the velocity,
$$\frac{d\mathbf v}{dt} = \frac{q}{\gamma m}\Big[\mathbf E + \mathbf v\times\mathbf B
  - \frac{\mathbf v\,(\mathbf v\cdot\mathbf E)}{c^2}\Big], \qquad \gamma = (1 - v^2/c^2)^{-1/2},$$
which is exactly $d(\gamma m\mathbf v)/dt = q(\mathbf E + \mathbf v\times\mathbf B)$, so it can be given to a
`CustomForce` and integrated with `rk4` (it depends on velocity, so not with `boris`).

Two fixes were found. A *synchrocyclotron* lowers the RF frequency as the bunch speeds up. An *isochronous* cyclotron
raises the field with radius, $B(r) = \gamma(r) B_0$ with $\gamma(r) = (1 - \omega_c^2 r^2/c^2)^{-1/2}$, so the
revolution frequency stays $\omega_c$ at every radius. (A real isochronous cyclotron also needs azimuthally varying
sectors to keep the beam focused vertically; this planar model leaves that out.) The extraction energy then follows
from the momentum $p = qB(R)R$: $T = \sqrt{p^2c^2 + m^2c^4} - mc^2$.

In [ ]:
w_c = qp * B / mp
period = 2 * np.pi / w_c                     # proton revolution period at low energy
peak_dee = V_dee / (math.sqrt(2 * math.pi) * gap)

def relativistic_cyclotron(isochronous, steps_per_turn=400, turns=200):
    def accel(t, pos, vel, mass):                # one proton: plain floats are faster than small arrays
        x, y = pos[0, 0], pos[0, 1]
        vx, vy, vz = vel[0]
        Ex = peak_dee * math.exp(-0.5 * (x / gap) ** 2) * math.cos(w_c * t)
        Bz = B / math.sqrt(1 - w_c**2 * (x * x + y * y) / c**2) if isochronous else B
        k = qp / mp * math.sqrt(1 - (vx * vx + vy * vy + vz * vz) / c**2)      # q / (gamma m)
        vE = vx * Ex / c**2
        return [[k * (Ex + vy * Bz - vx * vE), k * (-vx * Bz - vy * vE), -k * vz * vE]]

    w = ps.World("rk4")
    w.add_particle([0, 0, 0], mass=mp, charge=qp)
    w.add_force(ps.CustomForce(accel, name="relativistic Lorentz force"))
    extract = ps.Event(lambda t, pos, vel, mass: math.hypot(pos[0, 0], pos[0, 1]) - R, direction=+1, terminal=True)
    return w.run(period / steps_per_turn, steps_per_turn * turns, record_every=10, events=[extract], energies=False)

def kinetic_relativistic_MeV(vel, m):
    gamma = 1 / np.sqrt(1 - np.sum(vel**2, axis=-1) / c**2)
    return (gamma - 1) * m * c**2 / MeV

classic = relativistic_cyclotron(False)
iso = relativistic_cyclotron(True)
K_classic = kinetic_relativistic_MeV(classic.vel[:, 0], mp)
K_iso = kinetic_relativistic_MeV(iso.vel[:, 0], mp)
p_R = qp * B / math.sqrt(1 - (w_c * R / c) ** 2) * R
print(f"uniform field: peaks at {K_classic.max():.2f} MeV after {classic.t[K_classic.argmax()] / period:.0f} turns, "
      f"never reaches r = R (max r = {np.hypot(classic.pos[:, 0, 0], classic.pos[:, 0, 1]).max():.3f} m)")
print(f"isochronous:   extracted after {iso.t[-1] / period:.1f} turns with {K_iso[-1]:.3f} MeV; "
      f"theory {(math.sqrt((p_R * c) ** 2 + (mp * c**2) ** 2) - mp * c**2) / MeV:.3f} MeV")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
newton, _ = runs["proton"]
a1.plot(newton.t / period, kinetic_MeV(newton.vel[:, 0], mp), color=ps.plot.color(0), lw=1, label="Newtonian (section 3)")
a1.plot(classic.t / period, K_classic, color=ps.plot.color(1), lw=1, label="relativistic, uniform B")
a1.plot(iso.t / period, K_iso, color=ps.plot.color(2), lw=1, ls="--", label=r"relativistic, isochronous $B(r)$")
a1.set_xlabel("turns"); a1.set_ylabel("kinetic energy (MeV)")
ps.plot.tidy(a1, grid=True)

for k, (run, label) in enumerate([(classic, "relativistic, uniform B"), (iso, r"relativistic, isochronous $B(r)$")]):
    a2.plot(run.t / period, np.hypot(run.pos[:, 0, 0], run.pos[:, 0, 1]), lw=0.8, color=ps.plot.color(k + 1), label=label)
a2.axhline(R, color="0.5", ls=":", lw=1, label="extraction radius")
a2.set_xlabel("turns"); a2.set_ylabel("distance from centre (m)")
a2.set_title("uniform B: the proton falls out of step and spirals back in", fontsize=10)
ps.plot.tidy(a2, grid=True);

## 5. What the beam is for: Rutherford scattering

Before accelerators, the only fast projectiles were alpha particles from radioactive decay. In 1909–1913 Geiger and
Marsden fired them at gold foil and found that a few bounced almost straight back, which led Rutherford to the
nucleus. For a point nucleus of charge $Ze$ and an alpha of charge $ze$ and kinetic energy $T$, the deflection
$\theta$ at impact parameter $b$ satisfies
$$\tan\frac{\theta}{2} = \frac{d}{2b}, \qquad d = \frac{zZe^2}{4\pi\varepsilon_0 T},$$
where $d$ is the distance of closest approach in a head-on collision. Spreading the impact parameters uniformly over
the beam's area gives the Rutherford cross section
$d\sigma/d\Omega = (d/4)^2 / \sin^4(\theta/2)$.

Here the units are nuclear: lengths in fm, energies in MeV, masses in MeV/$c^2$, charges in $e$, so time is in fm/$c$
and the Coulomb constant is $e^2/4\pi\varepsilon_0 = 1.440$ MeV fm. The gold nucleus is a fixed point charge,
given as a `FieldForce`, so the alphas do not repel one another and many can share a world. The orbits combine a close
encounter on a scale of fm with a run-in of $10^5$ fm, so `run_adaptive` is the right tool.

In [ ]:
k_e = 1.43996448                       # e^2 / (4 pi eps0), MeV fm
m_alpha, z_alpha = IONS["alpha"][1], 2
Z_gold, T_alpha = 79, 5.0              # a 5 MeV alpha, as from radium's decay chain
d_min = k_e * z_alpha * Z_gold / T_alpha
v_alpha = math.sqrt(2 * T_alpha / m_alpha)     # units of c
print(f"closest approach in a head-on collision d = {d_min:.1f} fm; alpha speed {v_alpha:.4f} c")

def gold_nucleus(t, pos):
    r = np.linalg.norm(pos, axis=1, keepdims=True)
    return k_e * Z_gold * pos / r**3

def scatter(b, start=1e5):
    # alphas coming in along +x from x = -start at impact parameters b; returns deflection angles
    w = ps.World()
    for bi in b:
        w.add_particle([-start, bi, 0], [v_alpha, 0, 0], mass=m_alpha, charge=z_alpha)
    w.add_force(ps.FieldForce(E=gold_nucleus, name="gold nucleus"))
    t_end = 2 * start / v_alpha
    w.run_adaptive(t_end, rtol=1e-10, atol=1e-9, times=[t_end], energies=False)
    v = w.velocities
    return np.arctan2(np.hypot(v[:, 1], v[:, 2]), v[:, 0])

b = np.geomspace(1, 3000, 40)
theta = scatter(b, start=1e6)
theory = 2 * np.arctan(d_min / (2 * b))
print(f"deflection angles match 2 arctan(d / 2b) to {np.max(np.abs(theta / theory - 1)):.1e} (relative)")

In [ ]:
rng = np.random.default_rng(1909)
n_alpha, b_max = 20_000, 400.0                 # a uniform beam over a disc of radius 400 fm
b_beam = b_max * np.sqrt(rng.random(n_alpha))
theta_beam = scatter(b_beam)

edges = np.radians(np.linspace(15, 180, 23))
counts, _ = np.histogram(theta_beam, edges)
solid = 2 * np.pi * (np.cos(edges[:-1]) - np.cos(edges[1:]))
mid = 0.5 * (edges[:-1] + edges[1:])
flux = n_alpha / (np.pi * b_max**2)            # alphas per fm^2
measured = counts / (flux * solid)             # dsigma/dOmega in fm^2 / sr
th = np.radians(np.linspace(12, 179, 300))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
a1.loglog(b, np.degrees(theta), "o", ms=4, color=ps.plot.color(0), label="simulated")
a1.loglog(b, np.degrees(theory), color=ps.plot.color(1), lw=1.2, label=r"$2\arctan(d/2b)$")
a1.set_xlabel("impact parameter b (fm)"); a1.set_ylabel("deflection (deg)")
ps.plot.tidy(a1, grid=True)

a2.errorbar(np.degrees(mid), measured, yerr=np.sqrt(counts) / (flux * solid), fmt="o", ms=4,
            color=ps.plot.color(0), label=f"{n_alpha} simulated alphas")
a2.semilogy(np.degrees(th), (d_min / 4) ** 2 / np.sin(th / 2) ** 4, color=ps.plot.color(1), lw=1.2,
            label=r"Rutherford $(d/4)^2/\sin^4(\theta/2)$")
a2.set_xlabel(r"scattering angle $\theta$ (deg)"); a2.set_ylabel(r"$d\sigma/d\Omega$ (fm$^2$/sr)")
ps.plot.tidy(a2, grid=True)
print(f"{(theta_beam > np.pi / 2).sum()} of {n_alpha} alphas scattered backwards (theta > 90 deg)")

**Why this needs an accelerator.** A 5 MeV alpha turns round 45 fm from the centre of a gold nucleus, while the
nucleus itself has a radius of only about $1.2\,A^{1/3} \approx 7$ fm: the alpha never touches it, which is why the
pure Coulomb law describes the scattering so well. To reach the surface, the alpha has to climb the full Coulomb
barrier. The next cell estimates the barrier for a few targets.

In [ ]:
def coulomb_barrier(z1, a1, z2, a2, r0=1.2):
    # Coulomb energy (MeV) of two touching nuclei of radii r0 A^(1/3) fm
    return k_e * z1 * z2 / (r0 * (a1 ** (1 / 3) + a2 ** (1 / 3)))

for target, Z, A in [("lithium-7", 3, 7), ("nitrogen-14", 7, 14), ("copper-63", 29, 63), ("gold-197", 79, 197)]:
    print(f"{target:>12}: protons {coulomb_barrier(1, 1, Z, A):5.1f} MeV, alphas {coulomb_barrier(2, 4, Z, A):5.1f} MeV")

Light nuclei can be reached by the machines in sections 1–3; heavy ones need tens of MeV, which is where the
relativistic limit of section 4 starts to matter. (Cockcroft and Walton saw lithium split at well below its barrier
because protons can tunnel through it, a quantum effect that this classical engine does not model.)

**Where to go from here.**
- Add a `Coulomb` force between the protons of the linac bunch to see space charge spread it out.
- Model a synchrocyclotron by making the dee voltage's frequency a function of time, $\omega(t)$, that follows the
  bunch's falling revolution frequency.
- Let the gold nucleus recoil (unpin it, give it its mass, and use `Coulomb` with only the two particles) and compare
  with the centre-of-mass form of Rutherford's formula.